In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import numpy as np
import xesmf as xe
from workflow.scripts.utils import regrid_global
import pandas as pd
import matplotlib as mpl
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from workflow.scripts.plotting_tools import get_model_colordict

In [ ]:
def multi_model_mean(ds_dict, var_id, common_grid=None):
    dsets = []
    for modelN,ds in ds_dict.items():
        temp_ds = ds.mean(dim='year')
        temp_ds = temp_ds.drop(labels=['wavelength','height','member_id'],errors='ignore')
        temp_ds = regrid_global(temp_ds[[var_id]],ds_out=common_grid)
        dsets.append(temp_ds)
    # return dsets
    return xr.concat(dsets, dim='model'),xr.concat(dsets, dim='model').mean(dim='model'), xr.concat(dsets, dim='model').std(dim='model')

In [ ]:
def get_forcing(forcing_var: str,dataframes: dict):
    k = next(iter(dataframes))
    outdf = pd.DataFrame(index=dataframes.keys(), columns=dataframes[k].columns)
    for k,df in dataframes.items():
        try:
            outdf.loc[k,:] = df.loc[forcing_var]
        except KeyError:
            pass
            
    return outdf

In [ ]:
def mask_sign_agreement(ds, X):
    """
    Masks out grid points where at least X models agree on the sign.

    Parameters:
    ds (xr.Dataset): Input xarray dataset with dimensions 'models', 'lon', 'lat'.
    X (int): Threshold number of models that must agree on the sign to mask the grid point.

    Returns:
    xr.Dataset: Dataset with masked grid points.
    """
    # Calculate the sign of each model's data
    sign_data = np.sign(ds)

    # Sum the signs along the 'models' dimension
    sign_sum = sign_data.sum(dim='model')

    # Create a mask where the absolute value of the sign sum is greater than or equal to X
    mask = np.abs(sign_sum) >= X

    # Apply the mask to the dataset
    masked_ds = ds.where(mask)

    return mask

In [ ]:
order = [   
            'GISS-E2-1-G',
            'MIROC6',
            'GFDL-ESM4',
            'CNRM-ESM2-1',
            'UKESM1-0-LL',
            'IPSL-CM6A-LR-INCA',
            'NorESM2-LM',
            'MPI-ESM-1-2-HAM',
            'EC-Earth3-AerChem'   
        ]

In [ ]:
common_grid = xr.open_dataset(snakemake.input.common_grid)

In [ ]:
erf_paths = sorted(snakemake.input.gridded_ERF)
erft = {p.split('_')[-2]:xr.open_dataset(p).isel(year=slice(1, None)) for p in erf_paths}
dfs = {p.split('.')[0].split('_')[-1]: pd.read_csv(p,index_col=0) for p in snakemake.input.dust_forcing_table}
colors = get_model_colordict()
model_order = snakemake.params.get('model_order', order)

In [ ]:
ds_erft,mm_erft, std_erft = multi_model_mean(erft, 'ERFt', common_grid)
erft_glob_mean = get_forcing('ERFt', dfs)
erfsurf_glob_mean = get_forcing('ERFsurf',dfs)

In [ ]:
context_dict = {
    'axes.labelsize':8,
    'axes.spines.left': False,
    'axes.spines.right': False,
    'axes.spines.top': False,
    'axes.spines.bottom': True,
    "xtick.major.size" : 6,
    "xtick.minor.size" : 3.8,
    "xtick.major.width" : 1.2,
    "xtick.minor.width" : .8,
    "axes.linewidth" : .8,
    "xtick.labelsize" : 8
}


In [ ]:
agreement_mask = mask_sign_agreement(ds_erft, 7)['ERFt']

In [ ]:
mask_coarse = regrid_global(agreement_mask,lon=5,lat=5)

In [ ]:
mask_coarse = mask_coarse.drop('latitude_longitude')

In [ ]:
def plot_forcings_bar(df, nmodels, pos0, axes,colors,
                      dist=.9, spacing_frac=.01, scaling: pd.Series=None,
                     model_order: list=None):
    dx = dist/nmodels
    
    if scaling is not None:
        df = df.divide(scaling, axis=0)
    if model_order:
        df = df.loc[order]
    else:
        df = df.sort_values('diff')
    
    pos=pos0
    gap =  spacing_frac/dist
    n=0
    for model, series in df.iterrows():
        for axi in axes:
            
            if series['diff_sigificant'] == True:
                hatch='\\\\'
            else:
                hatch=None
#             print(pos0+(dx-gap)*.5)
            axi.barh(pos,series['diff'],height=dx-gap,zorder=100, facecolor=colors[model], 
                    xerr=series['st_error'],capsize=2, hatch=hatch)
#             print(pos0+(dx-gap)*.5, series['diff'])
            axi.plot(series['diff'],pos,  marker=".",  markerfacecolor=colors[model],
                    markeredgecolor= 'k',ms=10, zorder=300)
        
        if series['diff'] is not np.nan:
            n+=1
        pos+=dx


In [ ]:
with mpl.rc_context(context_dict):
    fig = plt.figure(figsize=(4.2*1.5, 4.5*1.5))
    
    # Create a GridSpec with different height ratios
    gs = fig.add_gridspec(2, 2, height_ratios=[2, 1])  # Allocate more space to the first subplot
    
    # Create the first subplot with a Cartopy projection
    ax1 = fig.add_subplot(gs[0, :], projection=ccrs.EckertIV())
    
    # Create the second subplot without a projection
    ax2 = fig.add_subplot(gs[1, 0])
     # Adjust the height space between the subplots
    ax3 = fig.add_subplot(gs[1,1])
    # Plotting on the first axis (map)
    ax1.coastlines()
    cmap = mpl.cm.get_cmap('RdYlBu_r').resampled(12)
    mm_erft['ERFt'].plot(ax=ax1, transform=ccrs.PlateCarree(), vmin=-3, vmax=3,cmap=cmap,
                         cbar_kwargs={'label':'ERF (W m-2)','orientation':'horizontal','shrink':0.7,'pad':0.035,
                                      'ticks':[-3.0,-2.5,-2.0,-1.5,-1.0,-0.5,0.0,0.5,1.0,1.5,2.0,2.5,3.0],'aspect':30,'extend':'both',
                                      'format':'%.1f'})
    # # agreement_mask.plot(ax=ax1, transform=ccrs.PlateCarree(),cmap=mpl.colors.ListedColormap(['none']), 
    #                     hatches=['none','x'], add_colorbar=True, edgecolors='k')



    lon, lat = np.meshgrid(mask_coarse['lon'], mask_coarse['lat'])
    ax1.contourf(lon, lat, mask_coarse.where(mask_coarse==True), transform=ccrs.PlateCarree(), hatches=['...'], alpha=0, colors='none')
    # ax1.pcolor(lon, lat, agreement_mask.where(agreement_mask==True), transform=ccrs.PlateCarree(), hatch=['..'])
    # ax1.pcolor(np.ma.array(agreement_mask.astype(int), mask = agreement_mask==True), cmap=mpl.colors.ListedColormap(['none']),
    #       hatch='xx', edgecolors='yellow', linewidth=0, transform=ccrs.PlateCarree())
    
    plot_forcings_bar(erft_glob_mean,9,0.1,[ax2],colors=colors,  model_order=model_order)
    # Plotting on the second axis
    plot_forcings_bar(erfsurf_glob_mean,9,0.1,[ax3],colors=colors,  model_order=model_order)

    legments = [
        Line2D([0],[0], markerfacecolor=colors[m], marker='o', label= m, color = 'w', markersize=10)
        for m in model_order[::-1]
    ]
    
    
    # legments.append(Line2D([0],[0], markerfacecolor='#FF005E', marker='*', label= 'Model mean', color = 'w',
    #                     markeredgecolor='k',markersize=20))

    # ax2.legend()
    fig.legend(handles=legments,ncol=3, bbox_to_anchor=[0.175, -0.02, 0.5, 0.5], loc='lower left',fontsize=7.5,
              frameon=False)

    ax1.text(-0.05, 1.0, 'a)', transform=ax1.transAxes, fontsize=12, fontweight='bold', va='top', ha='right')
    ax2.text(-0.05, 1.26, 'b)', transform=ax2.transAxes, fontsize=12, fontweight='bold', va='top', ha='right')
    ax3.text(-0.05, 1.26, 'c)', transform=ax3.transAxes, fontsize=12, fontweight='bold', va='top', ha='right')

    for axi in [ax2,ax3]:
        axi.yaxis.set_visible(False)
        axi.set_xlim(-1.2,0.25)
        axi.set_yticks([])
        axi.set_ylabel('')
        axi.axvline(0, color='slategray', linestyle='--', linewidth=2,zorder=-1)
        axi.set_xlabel('W m-2')
        ticks = axi.get_xticks()
        tick_labels = [f'{tick:.1f}' for tick in ticks]
        tick_labels = [label.replace('0.0', r'$\mathbf{0.0}$') for label in tick_labels]
        axi.set_xticklabels(tick_labels)

    

    mean = erft_glob_mean.mean(axis=0)
    diff_mean = mean['diff']
    ax2.set_title('Dust Forcing TOA', y=1.38)
    ax3.set_title('Dust Forcing Surface', y=1.38)

    ax2.axvline(diff_mean, ymax=0.7,color='black', linestyle='--', linewidth=1.5, zorder=2000)
    ax2.axvline(0, color='slategray', linestyle='--', linewidth=2,zorder=-1)
    ax2.annotate("$Ens_{mean}$\n" + f'{diff_mean:.2f} W/m²', xy=(diff_mean, 0.5), xytext=(diff_mean, 1.1),
                 arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=5),
                 fontsize=10, fontweight='bold', ha='center', zorder=2000)
    diff_mean_surf= erfsurf_glob_mean.mean(axis=0)['diff']
    ax3.axvline(diff_mean_surf, ymax=0.7,color='black', linestyle='--', linewidth=1.5, zorder=2000)
    erfsurf_glob_mean.mean(axis=0)['diff']
    ax3.annotate("$Ens_{mean}$\n" + f'{diff_mean_surf:.2f} W/m²', xy=(diff_mean_surf, 0.5), xytext=(diff_mean_surf, 1.1),
                 arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=5),
                 fontsize=10, fontweight='bold', ha='center', zorder=2000)
    pos2 = ax2.get_position()  # position of ax2
    ax2.set_position([pos2.x0, pos2.y0+0.02, pos2.width, pos2.height-0.05])
    pos3 = ax3.get_position()  # position of ax2
    ax3.set_position([pos3.x0, pos3.y0+0.02, pos3.width, pos3.height-0.05])
    plt.savefig(snakemake.output.outpath, dpi=300, bbox_inches='tight')
    plt.show()
    